# Task 4: Model Merging Research (SLERP)

Reproduce a basic SLERP merge between two CLIP checkpoints and evaluate the merged model against both parents.

- **Parent A:** generic CLIP `ViT-B-32` (`laion2b_s34b_b79k`) — the base model, unmodified.
- **Parent B:** the same base model, LoRA fine-tuned on a tiny RSICD subset (reproducing Task 3's recipe: 200 images, 1 epoch), then merged into a plain checkpoint via `merge_and_unload()`.
- **Merge method:** SLERP (spherical linear interpolation) applied per-tensor across the two parents' state dicts, swept over interpolation factor `t`. `mergekit` isn't used directly since it doesn't have first-class support for `open_clip`'s custom architecture — SLERP is implemented directly instead.
- **Evaluation:** the same held-out retrieval harness from Task 3, upgraded here to also report a *purity* score (fraction of top-3 retrievals matching the expected category), since Task 3 found the plain hit-rate metric ceilings out too easily to show small effects.
- **Flow:** load dataset → load parent A → reproduce parent B (LoRA fine-tune) → evaluation harness → sanity-check both parents → implement SLERP → merge at t=0.5 → sweep t → findings write-up (bottom of notebook).

### Step 1: Setup

In [ ]:
%pip install open_clip_torch peft
# Colab preinstalls an old torchao (0.10.0); peft>=0.15 requires torchao>=0.16.0 at import
# time even though this notebook never uses quantization (same fix as Task 3).
%pip install -U torchao

In [ ]:
import re
import copy
import random
from collections import defaultdict

import torch
import torch.nn.functional as F
import open_clip

from datasets import load_dataset
from peft import LoraConfig, get_peft_model

### Step 2: Load RSICD and define the fine-tuning subset + held-out evaluation sample

Same split as Task 3 (same seed), so parent B reproduces Task 3's fine-tune exactly and results are directly comparable.

In [ ]:
ds = load_dataset("arampacha/rsicd")
ds

In [ ]:
random.seed(0)

NUM_FT_SAMPLES = 200
ft_indices = random.sample(range(len(ds["train"])), NUM_FT_SAMPLES)
ft_indices_set = set(ft_indices)

CATEGORY_RE = re.compile(r'^([a-zA-Z]+)_\d+\.jpg$')

def category_from_filename(filename):
    name = filename.split('/')[-1]
    match = CATEGORY_RE.match(name)
    return match.group(1) if match else None

by_category = defaultdict(list)
uncategorized = 0
for idx, fname in enumerate(ds['train']['filename']):
    if idx in ft_indices_set:
        continue
    cat = category_from_filename(fname)
    if cat is None:
        uncategorized += 1
    else:
        by_category[cat].append(idx)

eval_sample_indices = []
for cat, idxs in by_category.items():
    eval_sample_indices.extend(random.sample(idxs, min(5, len(idxs))))

print(f"Fine-tuning subset: {len(ft_indices)} images")
print(f"Held-out eval sample: {len(eval_sample_indices)} images across {len(by_category)} labeled categories")

### Step 3: Load Parent A — base generic CLIP (kept pristine, never wrapped with LoRA)

In [ ]:
model_name = "ViT-B-32"

base_model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained="laion2b_s34b_b79k")
tokenizer = open_clip.get_tokenizer(model_name)
base_model = base_model.cuda().eval()

# Snapshot the pristine weights now, before anything else touches this architecture,
# so the merge step has a clean copy to work from regardless of what happens to base_model.
base_state_dict = {k: v.detach().clone().cpu() for k, v in base_model.state_dict().items()}

### Step 4: Reproduce Parent B — LoRA fine-tune (Task 3's recipe) on a separate model instance

A fresh model instance is used here (loaded from the `base_state_dict` snapshot) so `base_model` itself stays untouched as a clean Parent A.

In [ ]:
ft_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=None)
ft_model.load_state_dict(base_state_dict)
ft_model = ft_model.cuda()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["out_proj", "c_fc", "c_proj"],
    lora_dropout=0.05,
    bias="none",
)

lora_model = get_peft_model(ft_model, lora_config)
lora_model.print_trainable_parameters()
clip = lora_model.base_model.model

In [ ]:
BATCH_SIZE = 16
LR = 1e-4

optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=LR)

shuffled_ft_indices = ft_indices.copy()
random.shuffle(shuffled_ft_indices)

clip.train()
losses = []
num_batches = (len(shuffled_ft_indices) + BATCH_SIZE - 1) // BATCH_SIZE

for b, start in enumerate(range(0, len(shuffled_ft_indices), BATCH_SIZE), 1):
    batch_idx = shuffled_ft_indices[start:start + BATCH_SIZE]
    batch_images = torch.stack([preprocess(ds['train'][i]['image']) for i in batch_idx]).to("cuda")
    batch_captions = [ds['train'][i]['captions'][0] for i in batch_idx]
    batch_text = tokenizer(batch_captions).to("cuda")

    with torch.autocast("cuda"):
        image_features = clip.encode_image(batch_images)
        text_features = clip.encode_text(batch_text)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        logit_scale = clip.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.T
        logits_per_text = logits_per_image.T

        labels = torch.arange(len(batch_idx), device="cuda")
        loss = (F.cross_entropy(logits_per_image, labels) + F.cross_entropy(logits_per_text, labels)) / 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f"batch {b}/{num_batches}  loss={loss.item():.4f}")

print(f"\nMean loss over the epoch: {sum(losses) / len(losses):.4f}")

In [ ]:
# Bake the LoRA delta into plain Linear weights and drop the adapter wrapper, so Parent B
# ends up structurally identical to Parent A (same keys/shapes) — required for the SLERP merge.
finetuned_model = lora_model.merge_and_unload()
finetuned_model = finetuned_model.cuda().eval()

finetuned_state_dict = {k: v.detach().clone().cpu() for k, v in finetuned_model.state_dict().items()}

### Step 5: Evaluation harness (hit-rate + purity)

Same queries as Task 1/3, but now also reports **mean purity** (fraction of the top-3 that matches the expected category), since Task 3 found the plain hit-rate too coarse to see small effects.

In [ ]:
queries_with_expected = [
    ("a large bridge crossing a river", ["bridge"]),
    ("an airport with parked airplanes", ["airport"]),
    ("a dense green forest", ["forest"]),
    ("farmland divided into rectangular plots", ["farmland"]),
    ("a stadium with a running track", ["stadium"]),
    ("a residential area with many houses", ["denseresidential", "mediumresidential", "sparseresidential"]),
]
queries = [q for q, _ in queries_with_expected]

def evaluate_retrieval(model, label, verbose=True):
    model.eval()
    sample_images = [ds['train'][i]['image'] for i in eval_sample_indices]
    sample_filenames = [ds['train'][i]['filename'] for i in eval_sample_indices]
    sample_categories = [category_from_filename(f) for f in sample_filenames]

    image_tensors = torch.stack([preprocess(img) for img in sample_images]).to("cuda")

    with torch.no_grad(), torch.autocast("cuda"):
        bank_image_features = model.encode_image(image_tensors)
        bank_image_features /= bank_image_features.norm(dim=-1, keepdim=True)

        query_tokens = tokenizer(queries).to("cuda")
        query_features = model.encode_text(query_tokens)
        query_features /= query_features.norm(dim=-1, keepdim=True)

    sims = query_features @ bank_image_features.T

    if verbose:
        print(f"\n{'=' * 20} {label} {'=' * 20}")

    hits = 0
    purities = []
    for qi, (q, expected_cats) in enumerate(queries_with_expected):
        top_idx = sims[qi].topk(3).indices.tolist()
        top_cats = [sample_categories[ii] for ii in top_idx]
        hit = any(c in expected_cats for c in top_cats)
        hits += hit
        purity = sum(1 for c in top_cats if c in expected_cats) / len(top_cats)
        purities.append(purity)
        if verbose:
            mark = "\u2713" if hit else "\u2717"
            print(f"  {mark} {q!r} -> top3 categories: {top_cats}  (purity={purity:.2f})")

    mean_purity = sum(purities) / len(purities)
    if verbose:
        print(f"  Score: {hits}/{len(queries)}  |  Mean purity: {mean_purity:.3f}")
    else:
        print(f"{label}: score={hits}/{len(queries)}  mean_purity={mean_purity:.3f}")

    return hits, mean_purity

### Step 6: Sanity check — evaluate both parents before merging

In [ ]:
parent_a_score, parent_a_purity = evaluate_retrieval(base_model, "Parent A: Base CLIP (untuned)")
parent_b_score, parent_b_purity = evaluate_retrieval(finetuned_model, "Parent B: LoRA fine-tuned CLIP")

### Step 7: Implement SLERP merge

Applied per-tensor across the two parents' matching state-dict entries: each tensor is treated as a flat vector, normalized, and interpolated along the great-circle arc between the two directions, then rescaled. Falls back to linear interpolation when the two vectors are (nearly) parallel, since the SLERP formula is numerically unstable there (division by `sin(omega) ≈ 0`) — this also correctly handles the many tensors LoRA never touched, which are identical between the two parents.

In [ ]:
def slerp(t, v0, v1, eps=1e-8):
    orig_shape = v0.shape
    v0f = v0.flatten().double()
    v1f = v1.flatten().double()

    v0_norm = v0f.norm()
    v1_norm = v1f.norm()
    if v0_norm < eps or v1_norm < eps:
        return ((1 - t) * v0f + t * v1f).reshape(orig_shape).to(v0.dtype)

    v0_unit = v0f / v0_norm
    v1_unit = v1f / v1_norm
    dot = (v0_unit * v1_unit).sum().clamp(-1.0, 1.0)
    omega = torch.acos(dot)
    so = torch.sin(omega)

    if so.abs() < eps:  # nearly parallel (or identical) vectors -> plain linear interpolation
        result = (1 - t) * v0f + t * v1f
    else:
        result = (torch.sin((1 - t) * omega) / so) * v0f + (torch.sin(t * omega) / so) * v1f

    return result.reshape(orig_shape).to(v0.dtype)

def merge_state_dicts(t, state_dict_a, state_dict_b):
    merged = {}
    for key in state_dict_a:
        va, vb = state_dict_a[key], state_dict_b[key]
        if va.is_floating_point():
            merged[key] = slerp(t, va, vb)
        else:
            merged[key] = va.clone()  # non-float buffers (if any): keep Parent A's value
    return merged

### Step 8: Merge at t=0.5 and evaluate

In [ ]:
merged_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=None)
merged_state_dict = merge_state_dicts(0.5, base_state_dict, finetuned_state_dict)
merged_model.load_state_dict(merged_state_dict)
merged_model = merged_model.cuda().eval()

merged_score, merged_purity = evaluate_retrieval(merged_model, "SLERP merge (t=0.5)")

### Step 9: Sweep the interpolation factor `t`

`t=0.0` should reproduce Parent A exactly and `t=1.0` should reproduce Parent B exactly — useful as a correctness check on the SLERP implementation itself.

In [ ]:
sweep_results = []
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    sweep_state_dict = merge_state_dicts(t, base_state_dict, finetuned_state_dict)
    merged_model.load_state_dict(sweep_state_dict)
    score, purity = evaluate_retrieval(merged_model, f"t={t}", verbose=False)
    sweep_results.append((t, score, purity))

print("\nt      score    mean_purity")
for t, score, purity in sweep_results:
    print(f"{t:<6} {score}/{len(queries)}    {purity:.3f}")

## Step 10: Findings write-up

_TBD — to be filled in after running all cells and reviewing the actual output._